# Undiagnosed Hypertension and Diabetes Screening with NHANES
## Part 1: Download, harmonise and build the analysis dataset

Run each cell in order in Google Colab (Runtime > Run all also works).
Everything is downloaded directly from the CDC, so the analysis is fully reproducible.

**Design**
- Train: NHANES 2011-2012, 2013-2014, 2015-2016
- Validation: NHANES 2017 to March 2020 (pre-pandemic)
- Test: NHANES August 2021 to August 2023
- Adults aged 20 and over, examined at the mobile centre, not pregnant

**Outcomes** (each defined among people without a prior diagnosis of that condition)
- Undiagnosed diabetes: never told they have diabetes, not on insulin or diabetes pills, HbA1c >= 6.5% (ADA)
- Undiagnosed hypertension: never told they have high blood pressure, mean measured SBP >= 140 or DBP >= 90 mmHg
- Sensitivity versions: diabetes adding fasting glucose >= 126 mg/dL (fasting subsample); hypertension at 130/80 (2017 ACC/AHA)

### Step 1.1: Setup

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

RAW_DIR = "/content/nhanes_raw" if os.path.isdir("/content") else "./nhanes_raw"
OUT_DIR = "/content/nhanes_out" if os.path.isdir("/content") else "./nhanes_out"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print("Raw files will be saved in:", RAW_DIR)

### Step 1.2: Which files to download
Each cycle maps a component (demographics, body measures, blood pressure, ...) to its CDC file name.
From 2017 NHANES switched from manual (BPX) to oscillometric (BPXO) blood pressure readings,
and the 2021-2023 cycle dropped the family history of diabetes question and changed the physical activity questions.

In [ ]:
CYCLES = {
    "2011-2012": {"year": 2011, "split": "train", "files": {
        "demo": "DEMO_G", "bmx": "BMX_G", "bp": "BPX_G", "bpq": "BPQ_G", "diq": "DIQ_G", "ghb": "GHB_G",
        "glu": "GLU_G", "mcq": "MCQ_G", "smq": "SMQ_G", "paq": "PAQ_G", "rhq": "RHQ_G"}},
    "2013-2014": {"year": 2013, "split": "train", "files": {
        "demo": "DEMO_H", "bmx": "BMX_H", "bp": "BPX_H", "bpq": "BPQ_H", "diq": "DIQ_H", "ghb": "GHB_H",
        "glu": "GLU_H", "mcq": "MCQ_H", "smq": "SMQ_H", "paq": "PAQ_H", "rhq": "RHQ_H"}},
    "2015-2016": {"year": 2015, "split": "train", "files": {
        "demo": "DEMO_I", "bmx": "BMX_I", "bp": "BPX_I", "bpq": "BPQ_I", "diq": "DIQ_I", "ghb": "GHB_I",
        "glu": "GLU_I", "mcq": "MCQ_I", "smq": "SMQ_I", "paq": "PAQ_I", "rhq": "RHQ_I"}},
    "2017-2020": {"year": 2017, "split": "validation", "files": {
        "demo": "P_DEMO", "bmx": "P_BMX", "bp": "P_BPXO", "bpq": "P_BPQ", "diq": "P_DIQ", "ghb": "P_GHB",
        "glu": "P_GLU", "mcq": "P_MCQ", "smq": "P_SMQ", "paq": "P_PAQ", "rhq": "P_RHQ"}},
    "2021-2023": {"year": 2021, "split": "test", "files": {
        "demo": "DEMO_L", "bmx": "BMX_L", "bp": "BPXO_L", "bpq": "BPQ_L", "diq": "DIQ_L", "ghb": "GHB_L",
        "glu": "GLU_L", "mcq": "MCQ_L", "smq": "SMQ_L", "paq": "PAQ_L", "rhq": "RHQ_L"}},
}
BASE_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{name}.xpt"

### Step 1.3: Download (files are cached, so re-running is quick)

In [ ]:
def download_xpt(year, name, retries=3):
    path = os.path.join(RAW_DIR, f"{name}.xpt")
    if os.path.exists(path) and os.path.getsize(path) > 1000:
        return path
    url = BASE_URL.format(year=year, name=name)
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=120)
            if r.status_code == 200 and r.content[:13] == b"HEADER RECORD":
                with open(path, "wb") as f:
                    f.write(r.content)
                return path
            print(f"  {name}: unexpected response (status {r.status_code}), retrying")
        except Exception as e:
            print(f"  {name}: {e}, retrying")
        time.sleep(3)
    raise RuntimeError(f"Could not download {name} from {url}")

def load_cycle(cycle):
    info = CYCLES[cycle]
    raw = {}
    for comp, name in info["files"].items():
        path = download_xpt(info["year"], name)
        df = pd.read_sas(path, format="xport", encoding="latin1")
        # pandas reads a stored 0 in SAS transport files as a tiny float (about 5.4e-79); set those back to exactly 0
        num = df.select_dtypes("number").columns
        df[num] = df[num].mask(df[num].abs() < 1e-30, 0.0)
        raw[comp] = df
    return raw

RAW = {}
for cycle in CYCLES:
    print("Downloading", cycle)
    RAW[cycle] = load_cycle(cycle)
print("All files downloaded.")
for cycle, raw in RAW.items():
    print(cycle, {k: v.shape[0] for k, v in raw.items()})

### Step 1.4: Harmonise each cycle to one common set of variables

In [ ]:
def col(df, name):
    """Return a column if it exists, otherwise an all-missing column (so cycles that dropped a question still work)."""
    return df[name] if name in df.columns else pd.Series(np.nan, index=df.index)

def clean_code(s, valid):
    """Keep only valid answer codes; 7/9 (refused/don't know) and anything else become missing."""
    return s.where(s.isin(valid))

def harmonise(cycle, raw):
    d = raw["demo"].copy()
    out = pd.DataFrame({"SEQN": d["SEQN"].astype(int)})
    out["cycle"] = cycle
    out["split"] = CYCLES[cycle]["split"]
    out["age"] = col(d, "RIDAGEYR")
    out["sex_female"] = (col(d, "RIAGENDR") == 2).astype(float)
    out["race_eth"] = clean_code(col(d, "RIDRETH3"), [1, 2, 3, 4, 6, 7])
    out["education"] = clean_code(col(d, "DMDEDUC2"), [1, 2, 3, 4, 5])
    out["pir"] = col(d, "INDFMPIR")
    out["exam_status"] = col(d, "RIDSTATR")
    out["pregnant"] = (col(d, "RIDEXPRG") == 1).astype(float)
    out["wt_mec"] = col(d, "WTMECPRP") if "WTMECPRP" in d.columns else col(d, "WTMEC2YR")
    out["psu"] = col(d, "SDMVPSU")
    out["strata"] = col(d, "SDMVSTRA")
    out = out.set_index("SEQN")

    def add(comp, mapping):
        df = raw[comp].copy()
        df["SEQN"] = df["SEQN"].astype(int)
        df = df.set_index("SEQN")
        for new, old in mapping.items():
            out[new] = col(df, old).reindex(out.index)

    # Body measures
    add("bmx", {"weight_kg": "BMXWT", "height_cm": "BMXHT", "bmi": "BMXBMI", "waist_cm": "BMXWAIST"})
    out["whtr"] = out["waist_cm"] / out["height_cm"]

    # Blood pressure: mean of available readings (manual BPX before 2017, oscillometric BPXO after)
    bp = raw["bp"].copy()
    bp["SEQN"] = bp["SEQN"].astype(int)
    bp = bp.set_index("SEQN")
    if "BPXOSY1" in bp.columns:
        sys_cols, dia_cols, method = ["BPXOSY1", "BPXOSY2", "BPXOSY3"], ["BPXODI1", "BPXODI2", "BPXODI3"], "oscillometric"
    else:
        sys_cols, dia_cols, method = ["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"], ["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"], "auscultatory"
    sbp = bp[[c for c in sys_cols if c in bp.columns]]
    dbp = bp[[c for c in dia_cols if c in bp.columns]].replace(0, np.nan)  # a diastolic of 0 is not a valid reading
    out["sbp_mean"] = sbp.mean(axis=1).reindex(out.index)
    out["dbp_mean"] = dbp.mean(axis=1).reindex(out.index)
    out["bp_n_readings"] = sbp.notna().sum(axis=1).reindex(out.index).fillna(0)
    out["bp_method"] = method

    # Questionnaires
    add("bpq", {"told_htn": "BPQ020"})
    add("diq", {"told_dm": "DIQ010", "insulin": "DIQ050", "dm_pills": "DIQ070"})
    add("ghb", {"hba1c": "LBXGH"})
    add("glu", {"fpg": "LBXGLU"})
    add("mcq", {"famhx_dm": "MCQ300C", "chf": "MCQ160B", "chd": "MCQ160C", "mi": "MCQ160E", "stroke": "MCQ160F"})
    add("smq", {"smoked_100": "SMQ020", "smoke_now": "SMQ040"})
    add("rhq", {"gdm": "RHQ162"})

    # Physical activity: any moderate or vigorous leisure-time activity (questions changed in 2021-2023)
    pa = raw["paq"].copy()
    pa["SEQN"] = pa["SEQN"].astype(int)
    pa = pa.set_index("SEQN")
    if "PAD790Q" in pa.columns:
        mod = col(pa, "PAD790Q").where(col(pa, "PAD790Q") < 7777)
        vig = col(pa, "PAD810Q").where(col(pa, "PAD810Q") < 7777)
        active = ((mod > 0) | (vig > 0)).astype(float)
        active[mod.isna() & vig.isna()] = np.nan
    else:
        vig = clean_code(col(pa, "PAQ650"), [1, 2])
        mod = clean_code(col(pa, "PAQ665"), [1, 2])
        active = ((vig == 1) | (mod == 1)).astype(float)
        active[vig.isna() & mod.isna()] = np.nan
    out["active_leisure"] = active.reindex(out.index)
    sed = col(pa, "PAD680")
    out["sedentary_min"] = sed.where(sed < 7777).reindex(out.index)

    # Recode yes/no answers (1 = yes, 2 = no) to 1/0
    for c in ["told_htn", "insulin", "dm_pills", "famhx_dm", "chf", "chd", "mi", "stroke", "smoked_100", "gdm"]:
        out[c] = out[c].map({1: 1.0, 2: 0.0})
    out["told_dm"] = clean_code(out["told_dm"], [1, 2, 3])  # 3 = borderline
    out["cvd_history"] = out[["chf", "chd", "mi", "stroke"]].max(axis=1, skipna=True)
    out["smoking"] = np.select(
        [out["smoked_100"] == 0, (out["smoked_100"] == 1) & (out["smoke_now"] == 3),
         (out["smoked_100"] == 1) & out["smoke_now"].isin([1, 2])],
        ["never", "former", "current"], default=None)
    out["gdm"] = np.where(out["sex_female"] == 1, out["gdm"], 0.0)
    return out.reset_index()

frames = [harmonise(c, RAW[c]) for c in CYCLES]
data = pd.concat(frames, ignore_index=True)
print("Combined rows:", len(data))
print(data.groupby("cycle").size())

### Step 1.5: Eligibility and outcomes (prints the participant flow for the TRIPOD diagram)

In [ ]:
def build_cohort(data):
    flow = []
    d = data.copy()
    flow.append(("All NHANES participants", d.groupby("split").size()))
    d = d[d["age"] >= 20]
    flow.append(("Aged 20 and over", d.groupby("split").size()))
    d = d[d["exam_status"] == 2]
    flow.append(("Examined at the mobile centre", d.groupby("split").size()))
    d = d[d["pregnant"] != 1]
    flow.append(("Not pregnant", d.groupby("split").size()))

    # Diabetes
    d["dm_diagnosed"] = ((d["told_dm"] == 1) | (d["insulin"] == 1) | (d["dm_pills"] == 1)).astype(int)
    d["dm_at_risk_pop"] = ((d["dm_diagnosed"] == 0) & d["told_dm"].isin([2, 3]) & d["hba1c"].notna()).astype(int)
    d["undx_dm"] = np.where(d["dm_at_risk_pop"] == 1, (d["hba1c"] >= 6.5).astype(float), np.nan)
    fasting = d["fpg"].notna()
    d["undx_dm_fpg"] = np.where((d["dm_at_risk_pop"] == 1) & fasting,
                                ((d["hba1c"] >= 6.5) | (d["fpg"] >= 126)).astype(float), np.nan)

    # Hypertension
    d["htn_at_risk_pop"] = ((d["told_htn"] == 0) & d["sbp_mean"].notna() & d["dbp_mean"].notna()).astype(int)
    d["undx_htn"] = np.where(d["htn_at_risk_pop"] == 1,
                             ((d["sbp_mean"] >= 140) | (d["dbp_mean"] >= 90)).astype(float), np.nan)
    d["undx_htn_130"] = np.where(d["htn_at_risk_pop"] == 1,
                                 ((d["sbp_mean"] >= 130) | (d["dbp_mean"] >= 80)).astype(float), np.nan)
    flow.append(("Diabetes analysis: no prior diagnosis, HbA1c measured", d[d["dm_at_risk_pop"] == 1].groupby("split").size()))
    flow.append(("Hypertension analysis: no prior diagnosis, BP measured", d[d["htn_at_risk_pop"] == 1].groupby("split").size()))
    flow_table = pd.DataFrame({name: counts for name, counts in flow}).T[["train", "validation", "test"]]
    return d, flow_table

cohort, flow_table = build_cohort(data)
print(flow_table)

### Step 1.6: Weighted prevalence of the outcomes in each split
Survey weights are divided by the number of cycles pooled, following NCHS guidance, so prevalence
reflects the US adult population rather than the raw sample.

In [ ]:
n_cycles = cohort.groupby("split")["cycle"].nunique()
cohort["wt_pooled"] = cohort["wt_mec"] / cohort["split"].map(n_cycles)

def weighted_prev(df, outcome):
    d = df[df[outcome].notna() & df["wt_pooled"].notna()]
    return pd.Series({
        "n": len(d),
        "cases": int(d[outcome].sum()),
        "unweighted_%": round(100 * d[outcome].mean(), 2),
        "weighted_%": round(100 * np.average(d[outcome], weights=d["wt_pooled"]), 2),
    })

prev_rows = []
for outcome in ["undx_dm", "undx_dm_fpg", "undx_htn", "undx_htn_130"]:
    for split in ["train", "validation", "test"]:
        r = weighted_prev(cohort[cohort["split"] == split], outcome)
        r["outcome"], r["split"] = outcome, split
        prev_rows.append(r)
prevalence = pd.DataFrame(prev_rows)[["outcome", "split", "n", "cases", "unweighted_%", "weighted_%"]]
print(prevalence.to_string(index=False))

### Step 1.7: Missing data check for the candidate predictors

In [ ]:
PREDICTORS = ["age", "sex_female", "race_eth", "education", "pir", "bmi", "waist_cm", "height_cm", "weight_kg",
              "whtr", "sbp_mean", "dbp_mean", "told_htn", "told_dm", "famhx_dm", "gdm", "cvd_history",
              "smoking", "active_leisure", "sedentary_min"]
missing = (cohort.groupby("split")[PREDICTORS].apply(lambda g: g.isna().mean() * 100).T.round(1))[["train", "validation", "test"]]
print("Percent missing by split (100% means the question was not asked in that cycle):")
print(missing)

### Step 1.8: Table 1, characteristics of each analysis population

In [ ]:
def table1(df, pop_flag):
    d = df[df[pop_flag] == 1]
    rows = {}
    for split in ["train", "validation", "test"]:
        s = d[d["split"] == split]
        rows[split] = {
            "N": len(s),
            "Age, mean (SD)": f"{s['age'].mean():.1f} ({s['age'].std():.1f})",
            "Female, %": f"{100 * s['sex_female'].mean():.1f}",
            "BMI, mean (SD)": f"{s['bmi'].mean():.1f} ({s['bmi'].std():.1f})",
            "Waist cm, mean (SD)": f"{s['waist_cm'].mean():.1f} ({s['waist_cm'].std():.1f})",
            "SBP mmHg, mean (SD)": f"{s['sbp_mean'].mean():.1f} ({s['sbp_mean'].std():.1f})",
            "Current smoker, %": f"{100 * (s['smoking'] == 'current').mean():.1f}",
            "Active in leisure time, %": f"{100 * s['active_leisure'].mean():.1f}",
            "Family history of diabetes, %": f"{100 * s['famhx_dm'].mean():.1f}" if s['famhx_dm'].notna().any() else "not asked",
        }
    return pd.DataFrame(rows)

print("Diabetes analysis population")
print(table1(cohort, "dm_at_risk_pop"))
print("\nHypertension analysis population")
print(table1(cohort, "htn_at_risk_pop"))

### Step 1.9: Save the analysis dataset
This file is your harmonised NHANES dataset. Parts 2 onward load it, so you only download once.

In [ ]:
# Sanity check: this version fixes the zero-value bug, so leisure activity in 2021-2023 must not be 100%
pct_active_test = 100 * cohort.loc[cohort["split"] == "test", "active_leisure"].mean()
print(f"Active in leisure time, 2021-2023: {pct_active_test:.1f}%")
assert pct_active_test < 95, "Old notebook detected: please use the FIXED Part 1"
assert (cohort["dbp_mean"].dropna() > 1).all(), "Diastolic zeros still present"
print("Sanity check passed: fixed version")

keep = ["SEQN", "cycle", "split", "wt_mec", "wt_pooled", "psu", "strata", "bp_method",
        "dm_at_risk_pop", "undx_dm", "undx_dm_fpg", "hba1c", "fpg",
        "htn_at_risk_pop", "undx_htn", "undx_htn_130"] + [p for p in PREDICTORS if p not in ["told_htn", "told_dm"]] + ["told_htn", "told_dm"]
analytic = cohort[keep]
csv_path = os.path.join(OUT_DIR, "nhanes_analytic.csv")
analytic.to_csv(csv_path, index=False)
flow_table.to_csv(os.path.join(OUT_DIR, "participant_flow.csv"))
prevalence.to_csv(os.path.join(OUT_DIR, "prevalence.csv"), index=False)
missing.to_csv(os.path.join(OUT_DIR, "missingness.csv"))
print("Saved:", csv_path, analytic.shape)

try:
    from google.colab import files
    files.download(csv_path)
    files.download(os.path.join(OUT_DIR, "participant_flow.csv"))
    files.download(os.path.join(OUT_DIR, "prevalence.csv"))
except Exception:
    print("Not running in Colab, files saved locally.")

### What to send back
Copy the printed output of Steps 1.5, 1.6 and 1.7 (participant flow, prevalence and missing data)
into the chat, or send the three CSV files. Part 2 (features, baselines and models) is built on these numbers.